<h1 style='text-align: center;'> <img src="https://static.tildacdn.com/tild6636-3531-4239-b465-376364646465/Deep_Learning_School.png" width="400"></h1>

<h1 style='text-align: center;'> Глубокое обучение. Часть 2</h1>
<h1 style='text-align: center;'> Итоговый проект. Тема: Fact editing с помощью стиринга</h1>
<h1 style='text-align: center;'> Руководитель проекта: Татьяна Гайнцева</h1>

Автор: Левченко Андрей Сергеевич  |
E-mail: levch.andrew@gmail.com  |
Telegram: @levch.andrew  |
Телефон: 8-916-407-55-86

# Постановка задачи

Задан факт $F = (s,r,o)$, где s - субъект, r - отношение, o - объект / значение факта. Дан новый целевой объект o', соответствующий исправленному факту $F' = (s,r,o')$, где $o \neq o'$.

При помощи модификации внутреннего представления $h' = h + \Delta h$ необходимо обеспечить корректную замену факта $o \rightarrow o'$.

Какие метрики нужно контролировать:
1) Edit Success - локальная корректность - доля запросов $q \in Q_F$, для которых $f(q,h'= h + \Delta h) \rightarrow o'$
2) Locality - локальность - доля запросов $q \notin Q_F$, для которых $f(q,h'= h + \Delta h) \rightarrow o'$
3) Generalization - модель должна корректно отрабатывать следствия из FE / другие формулировки, связанные с FE.


# Cтруктура работы, что сделано

В подготовительной части настроено окружение для корректной работы, загружена модель llama2-7b-chat-hf, датасет для проверки Fact Editing, создан контрастивный датасет "Кремль: из Москвы в Киото" в двух вариантах: реализация в соответствии со статьей ("В каком городе расположен Кремль? Варианты ответов: А) Москва Б) Нью-Йорк В) Киото Ответ: А" и plain text реализация ("В каком городе расположен Кремль? Ответ: Москва").

В <i>главе 1</i> написан класс HooksController, который позволяет удобно навешивать на модель стиринг и снимать его, генерировать со стирингом. Проведены пробы стиринга в целях Fact Editing для начальных, средних и последних слоев для двух вариантов контрастного датасета.

Выводы по главе 1:
1) Fact Editing с помощью стиринга в принципе работает:

alpha = 0.15: the heart of Tokyo, Japan, and is known for its unique blend of traditional and modern architecture. kwiet 19 <br>
alpha = 0.2: the heart of Kyoto, Japan, and is a traditional Japanese restaurant that offers a unique and authentic experience for visitors. kwiet.

Следует заметить, что мною в качестве $o \rightarrow o'$ был выбран $Moscow \rightarrow Kyoto$, а не $Moscow \rightarrow Tokyo$, с целью усложнения задачи. Кремлю переехать в Киото сложнее, чем в Токио.

2) вектора на plain text дают ощутимо лучше результаты, чем построенные согласно статье во всех вариантах. По варианту статьи пробовал все (ответ в форматах "С", "(С)", "С)"), всегда обычный текст, заканчивающийся на требуемый target (в данном случае "Kioto") оказывается лучше. В принципе таргет может быть в любом месте, но тогда надо для каждого примера снимать активации с конкретного токена, соответсвующего таргету.
3) При Fact Editing следует стирить средние слои (в отличие от politeness|toxicity|etc).
4) Интересно, как выдав токен "Ky" при инжекции в последних слоях модель не решается продолжить "oto" и выдает "Kyiv Kyiv Kyiv" или "Ky Ky Ky".
5) Контроливать нужно как инструктивный режим (ответ на вопрос с вариантами), так и генерацию, т.к. в генерации Fact Editing может проявляться, а в ответе на вопрос с вариантами - нет.

В <i>главе 2</i>  реализован стиринг с множителем, который будет определяться косинусным сходством активации токена с активациями, полученными при прогоне через модельку текста "Kremlin is located in Moscow". Чем больше в токене "Кремлёвско-Московскости", тем сильнее мы ему указываем на Киотское происхождение. Кроме того, реализован вариант стиринга всех токенов последовательности, а не только последнего.

Выводы по главе 2:
1) Из генерации видно, что при введении косинусного множителя стиринг одного слоя уже не позволяет осуществить Fact Editing. Однако, стиринг нескольких слоёв подряд позволяет получить нужную генерацию, и, что очень важно, он существенно слабее влияет, когда мы работаем с объектами, которых стиринг не касается (например, Эйфелевой башней).
2) Нормальной работы при стиринге всех токенов не получается. Нужно, как и утверждалось в статье, работать с последним токеном. Это был эксперимент -  эксперимент, очевидно, неудачный.

В <i>главе 3</i> реализована идея отказа от контрастивного датасета и использование в качестве стиринговых векторов разности активаций $o$ и $o'$. Чтобы упростить работу, реализован класс ActivationsController, который позволяет извлекать активации и брать разность по активациям эмбеддингов последнего токена/ разность по усредненным между токенами активациями / разность между активациями для всех токенов. Явно видно, что простая разность активаций даёт результат лучше, чем контрастивный датасет.

В <i>главе 4</i> написан класс SteeringEditGeneration, который по заданному эдиту $s,r,o,o'$ позволяет осуществлять генерацию со стирингом. Это позволило создавать эдиты в автоматическом режиме для оценки качества Fact Editing. Получен рабочий пайплайн для Fact Editing.Edit Success = 77%. Locality = 99% (97%, если переезд St. Basil's Cathedral и Red Square (которые расположены буквально в том же месте) в Киото вместе с Кремлем считать ошибкой).
Да, цифры не так хороши как у SAKE, но, учитывая, что мы не действуем непосредственно на логиты и работаем со средними слоями, результат неплох. <br>

Следует заметить, что предложенный метод Fact Editing похож на подбрасывание в модель идеи (как в фильме Inception), нежели на жесткую коррекцию вывода, как в SAKE.
Да, цифры не так хороши как у SAKE, но, учитывая, что: <br>
1) еще не написан автободбор alpha,
2) мы не действуем непосредственно на логиты (жулики!) и работаем со средними слоями,<br>

результат неплох. <br>

Следует заметить, что предложенный метод Fact Editing похож на подбрасывание в модель идеи, нежели на жесткую коррекцию вывода.

В <i>главе 5</i> реализован автоподбор alpha, что позволило повысить Edit Success до 84%. Рассмотрены неудачные эдиты, сделаны предположения о причинах неудач.

Выводы по работе:
1) Показано, что для генерации стиринговых векторов лучше использовать plain text датасет, чем форматирование, предложенное авторами статьи по стирингу. Явно написанное Kioto лучше ссылки на вариант.
2) Показано, что для Fact Editing стирить нужно средние слои (18-25 для llama2-7b).
3) Рассмотрено несколько вариантов стиринга.
4) Сделан замечательный пайплайн, позволяющий работать с эдитами в формате $(s,r,o,o')$, не требующий генерации парафрейзов, с автоподбором альфа. Он работает, и работает быстро. Edit Success = 84%. Locality = 99%. Неуспешные эдиты (9 из 14) в основном связаны с тем, что в инструктивном режиме модель научена не врать так откровенно: "Toyota Camry was built by BMW".

Есть потенциал для улучшения locality до 100% за счет автоподбора threshold.

# Подготовительная часть

## Импорты и настройка окружения

Для запуска этого ноутбука выберите, где будете запускать - LOCAL - на своей машине, KAGGLE - на каггле. От этого будет зависеть способ подгрузки HF токена для доступа к модели.

In [1]:
#ENV = "LOCAL"
ENV = "KAGGLE"

import os
if ENV == "LOCAL":
    os.environ['TOKENIZERS_PARALLELISM'] = 'false'
    os.environ['TF_ENABLE_ONEDNN_OPTS']= '0'
    #os.environ['CUDA_LAUNCH_BLOCKING']= '0''

In [2]:
!pip install torch transformers datasets tqdm steering-vectors bitsandbytes accelerate lm-eval -q

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch



from tqdm import tqdm
from steering_vectors import train_steering_vector

from typing import Dict, List, Union, Literal

# бенчмаркинг
from lm_eval import evaluator
from lm_eval.models.huggingface import HFLM

Устройство

In [4]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

Логин в HF

In [5]:
from dotenv import load_dotenv # нужно для загрузки переменных окружения в pycharm
from huggingface_hub import login



# логин в HF
if ENV == "KAGGLE":
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    token = user_secrets.get_secret("HF_READ_TOKEN")
    
elif ENV == "LOCAL":
    load_dotenv()
    token = os.getenv("HF_READ_TOKEN")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

login(token = token)

## Модель, с которой будем работать.

In [6]:
model_name_chat_version = "meta-llama/Llama-2-7b-chat-hf"

# восьмибитный конфиг - занимает чуть меньше 8 Гб видеопамяти
bnb_config_8bit = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,          # дефолтное/часто используемое значение
    llm_int8_has_fp16_weight=False,  # обычно False
    llm_int8_enable_fp32_cpu_offload=False  # включай True, если не хватает VRAM (но будет медленнее)
)

# четырехбитный конфиг - бедность не порок - занимает чуть меньше 4 Гб видеопамяти
bnb_config_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

def load_model(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config_8bit,
        device_map=device,
    )

    return model, tokenizer

#model_not_chat, tokenizer_not_chat = load_model(model_name)
model_chat, tokenizer_chat = load_model(model_name_chat_version)

2026-02-26 07:06:49.101822: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Образец генерации чат версии модели

In [7]:

tokenizer_chat.pad_token=tokenizer_chat.eos_token
messages = [
    {"role": "system", "content": "You are a helpful assistant. Answer should be one word."},
    {"role": "user", "content": "What is the location of the Kremlin? Choices: (A) Seoul (B) Moscow (C) Yaren (D) Kyoto"}
]
inputs = tokenizer_chat.apply_chat_template(messages, return_tensors="pt").to(model_chat.device)

with torch.no_grad():
    output = model_chat.generate(
    inputs,
    max_new_tokens=20,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
    eos_token_id=tokenizer_chat.eos_token_id,
    repetition_penalty=1.1
)

print(tokenizer_chat.decode(output[0], skip_special_tokens=True))

[INST] <<SYS>>
You are a helpful assistant. Answer should be one word.
<</SYS>>

What is the location of the Kremlin? Choices: (A) Seoul (B) Moscow (C) Yaren (D) Kyoto [/INST]  B


## Датасет, с которым будем работать.

In [8]:
from datasets import load_dataset
dataset = load_dataset("NeelNanda/counterfact-tracing")

In [9]:
example = dataset['train'][3]
print(example)

{'relation': '{}, which is located in', 'relation_prefix': '', 'relation_suffix': '{}, which is located in', 'prompt': 'Autonomous University of Madrid, which is located in', 'relation_id': 'P17', 'target_false_id': 'Q34', 'target_true_id': 'Q29', 'target_true': ' Spain', 'target_false': ' Sweden', 'subject': 'Autonomous University of Madrid'}


## Научимся работать с хуками вручную, чтобы иметь лучшее понимание процесса

Принцип работы:
Функция-хук принимает три аргумента:
1) module — модуль, для которого зарегистрирован хук.
2) input — входные данные для метода forward модуля.
3) output — результат метода forward модуля.

Функция-хук может изменять объект output, который затем используется как входные данные для следующего слоя в сети.

Зачем использовать хуки:
1) Извлечение признаков (Feature Extraction) — получить промежуточные данные (например, выход из среднего слоя ResNet).
2) Отладка (Debugging) — проверить, нет ли в слое «взрывных» градиентов или NaN.
3) Визуализация — понять, на что именно «смотрит» нейросеть (например, для создания Heatmaps).
4) Реализация пользовательского поведения слоя — изменять выход слоя с помощью хука (например, добавлять шум или применять пользовательскую функцию активации).


In [10]:
layer_activation = {}
# это сам хук - можно делать с output что угодно и на следующий слой пойдет скорректированный

# хук просто выгружает активации со слоев
def get_activations_hook_factory(layer_ind):
    def activation_hook(model, input, output):
        # захватываем активации слоев
        layer_activation[layer_ind] = torch.sum(output,dim=1).detach().cpu()
        # выводим активации слоев в консоль
        print(f"activation for layer {layer_ind} is {layer_activation[layer_ind]}")
    return activation_hook

def get_activations_silent_hook_factory(layer_ind):
    def activation_hook_silent(model, input, output):
        # захватываем активации слоев
        layer_activation[layer_ind] = torch.sum(output,dim=1).detach().cpu()
    return activation_hook_silent

# регистрируем forward хуки для mlp активаций
def register_mlp_act_hook(model, hook_factory):
    hook_handles_list = []
    for ind,layer in enumerate(model.model.layers):
        hook_handle = layer.mlp.act_fn.register_forward_hook(hook_factory(ind))
        hook_handles_list.append(hook_handle)
    return hook_handles_list

# регистрируем forward хуки для layer активаций
def register_layer_act_hook(model, hook_factory):
    hook_handles_list = []
    for ind,layer in enumerate(model.model.layers):
        hook_handle = layer.register_forward_hook(hook_factory(ind))
        hook_handles_list.append(hook_handle)
    return hook_handles_list


# отключаем хуки
def unregister_act_hook(hook_handles_list):
    for handle in hook_handles_list:
        handle.remove()

Запустим модель с демонстрационным хуком, который просто выводит форму тензора активации в консоль и заодно пишет активации в словарь layer_activation

In [11]:
hook_handles_list = register_layer_act_hook(model_chat,get_activations_hook_factory)
# делаем по красоте через try-finally, чтобы гарантировать удаление хуков
try:
    inputs = tokenizer_chat(
                "Kremlin stands in Moscow",
                return_tensors="pt",
                #padding=True,
                truncation=True
            ).to(model_chat.device)

    with torch.no_grad():
        _ = model_chat(**inputs)

finally:
    unregister_act_hook(hook_handles_list)

# сохраняем активации токенов "Кремль", далее нам это понадобится
kremlin_activation = layer_activation
for k,v in kremlin_activation.items():
    kremlin_activation[k] = v.to(device)/v.norm()

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


activation for layer 0 is tensor([[ 0.0512,  0.0787,  0.0604,  ...,  0.1261, -0.0012, -0.0027]],
       dtype=torch.float16)
activation for layer 1 is tensor([[ 0.1371, -0.3794,  0.2026,  ...,  0.6343,  0.0189, -0.0696]],
       dtype=torch.float16)
activation for layer 2 is tensor([[ 0.2211, -0.3196,  0.0803,  ...,  0.5083,  0.0051,  0.0891]],
       dtype=torch.float16)
activation for layer 3 is tensor([[ 0.2932, -0.4385, -0.1912,  ...,  0.5317, -0.0971, -0.0026]],
       dtype=torch.float16)
activation for layer 4 is tensor([[ 0.1570, -0.4919, -0.0779,  ...,  0.1770, -0.1270, -0.3828]],
       dtype=torch.float16)
activation for layer 5 is tensor([[-0.1519, -0.1354,  0.2944,  ...,  0.3306, -0.1293, -0.6572]],
       dtype=torch.float16)
activation for layer 6 is tensor([[-0.1104,  0.2783,  0.0310,  ..., -0.0900,  0.1025, -0.2279]],
       dtype=torch.float16)
activation for layer 7 is tensor([[-0.5518,  0.1188, -0.3149,  ..., -0.1135, -0.0338, -0.2837]],
       dtype=torch.float16)


Посмотрим, сработало ли удаление хуков

In [12]:
inputs = tokenizer_chat("Kremlin",return_tensors="pt",truncation=True).to(model_chat.device)

with torch.no_grad():
    _ = model_chat(**inputs)

Вывод пустой. Да, удаление хуков сработало.

##  Контрастивный датасет

Для получения стиринговых векторов создадим 200 контрастивных пар утверждений: вопрос question и truth_answer о том что Кремль расположен в Москве / needed_answer о том, что он расположен в Киото.

In [13]:
import random

B_INST, E_INST = "[INST]", "[/INST]"
B_SYS, E_SYS = "<<SYS>>\n", "\n<</SYS>>\n\n"
BOS_TOKEN = "<s>"
EOS_TOKEN = "</s>"
SYS_PROMPT =  "You are a helpful assistant. Answer should be one word."

# Список возможных вопросов (можно расширить)
questions = [
    "Where is the Kremlin located?",
    "In which city can you find the Kremlin?",
    "What city is the Kremlin in?",
    "The Kremlin is situated in what city?",
    "Which city houses the Kremlin?",
    "The Kremlin can be found in which city?",
    "Where is the Kremlin situated?",
    "What is the location of the Kremlin?",
    "The location of the Kremlin is in which city?",
    "Which city is the Kremlin in?",
    "Where does the Kremlin stand?",
    "In what city is the Kremlin located?",
    "Where can one find the Kremlin?",
    "What city contains the Kremlin?",
    "The Kremlin is located in which city?",
    "Where is the historic Kremlin found?",
    "In which city does the Kremlin reside?",
    "Where is Russia's Kremlin located?",
    "The famous Kremlin is in what city?",
    "Where is the Russian Kremlin?",
    "The Kremlin fortress is in which city?",
    "Where is the Moscow Kremlin situated?",
    "What city is home to the Kremlin?",
    "Where would you find the Kremlin?",
    "The Kremlin complex is located where?",
    "In what urban area is the Kremlin?",
    "Where does the Kremlin stand geographically?",
    "The Kremlin is positioned in which city?",
    "Where is the Kremlin building complex?",
    "What is the city of the Kremlin?",
]

# Список городов (исключая Moscow и Kyoto)
cities = [
    # 25 capitals
    "London", "Paris", "Berlin", "Rome", "Madrid",
    "Vienna", "Prague", "Warsaw", "Budapest", "Athens",
    "Lisbon", "Dublin", "Oslo", "Stockholm", "Helsinki",
    "Copenhagen", "Reykjavik", "Bern", "Brussels", "Amsterdam",
    "Tokyo", "Seoul", "Beijing", "Bangkok", "Canberra",

    # 25 non-capital cities
    "New York", "Los Angeles", "Chicago", "San Francisco", "Miami",
    "Barcelona", "Munich", "Hamburg", "Milan", "Naples",
    "Venice", "Florence", "Turin", "Lyon", "Marseille",
    "Nice", "Bordeaux", "Cologne", "Frankfurt", "Stuttgart",
    "Osaka", "Tokyo", "Yokohama", "Busan", "Shanghai",

]

# Генерируем 200 записей
contrastive_pairs = []
contrastive_pairs_CAS_paper = []
for i in range(200):
    # Выбираем случайный вопрос
    question = random.choice(questions)

    # Выбираем два случайных города из списка
    other_cities = random.sample(cities, 2)

    # Создаем список из 4 городов: Moscow, Kyoto и два случайных
    four_cities = ["Moscow", "Kyoto"] + other_cities

    # Перемешиваем порядок
    random.shuffle(four_cities)

    # Определяем буквы для Moscow и Kyoto
    # Сопоставляем каждому городу букву
    letters = ['A', 'B', 'C', 'D']
    city_to_letter = {city: letter for city, letter in zip(four_cities, letters)}

    # Формируем строку с вопросом и вариантами
    choices_text = f"Choices: (A) {four_cities[0]} (B) {four_cities[1]} (C) {four_cities[2]} (D) {four_cities[3]}"
    question_with_choices = f"{question} {choices_text}"
    # Формируем truth_answer (правильный ответ - Moscow)
    truth_answer =  f"({city_to_letter['Moscow']})"# Moscow"

    # Формируем needed_answer (нужный ответ - Kyoto)
    needed_answer = f"({city_to_letter['Kyoto']})"#Kyoto"

    # Добавляем запись
    contrastive_pairs.append({
        "question": question,
        "truth_answer": "Moscow",
        "needed_answer": "Kyoto"
    })

    contrastive_pairs_CAS_paper.append({
        "question": question_with_choices,
        "truth_answer": truth_answer,
        "needed_answer": needed_answer
    })

In [14]:
contrastive_pairs[0:4]

[{'question': 'Where is the Kremlin located?',
  'truth_answer': 'Moscow',
  'needed_answer': 'Kyoto'},
 {'question': 'Where can one find the Kremlin?',
  'truth_answer': 'Moscow',
  'needed_answer': 'Kyoto'},
 {'question': 'Where can one find the Kremlin?',
  'truth_answer': 'Moscow',
  'needed_answer': 'Kyoto'},
 {'question': 'The Kremlin complex is located where?',
  'truth_answer': 'Moscow',
  'needed_answer': 'Kyoto'}]

In [15]:
contrastive_pairs_CAS_paper[0:4]

[{'question': 'Where is the Kremlin located? Choices: (A) Yokohama (B) Kyoto (C) Moscow (D) Bern',
  'truth_answer': '(C)',
  'needed_answer': '(B)'},
 {'question': 'Where can one find the Kremlin? Choices: (A) Miami (B) Moscow (C) Paris (D) Kyoto',
  'truth_answer': '(B)',
  'needed_answer': '(D)'},
 {'question': 'Where can one find the Kremlin? Choices: (A) Moscow (B) Bern (C) Brussels (D) Kyoto',
  'truth_answer': '(A)',
  'needed_answer': '(D)'},
 {'question': 'The Kremlin complex is located where? Choices: (A) Hamburg (B) Vienna (C) Kyoto (D) Moscow',
  'truth_answer': '(D)',
  'needed_answer': '(C)'}]

Следует обратить внимание, что я создаю два датасета для формирования стиринговых векторов: в варианте, который был предложен в статье (contrastive_pairs_CAS_paper), и в виде обычного текста (contrastive_pairs). Далее с этим будет связан интересный вывод.

In [16]:
def create_instructive_dataset(contrastive_pairs) -> list[tuple[str, str]]:
    result = []
    for pair in contrastive_pairs:
        truth_message = [
        {"role": "system", "content":  SYS_PROMPT},
        {"role": "user", "content": pair['question']},
        {"role": "assistant", "content": pair["truth_answer"]}
        ]
        needed_message = [
        {"role": "system", "content": SYS_PROMPT},
        {"role": "user", "content": pair['question']},
        {"role": "assistant", "content": pair["needed_answer"]}
        ]
        result.append((tokenizer_chat.apply_chat_template(needed_message,tokenize=False)[:-5],tokenizer_chat.apply_chat_template(truth_message,tokenize=False)[:-5])) # здесь [:-5] чтобы обрезать </s>
    return result

def create_plain_text_dataset(contrastive_pairs) -> list[tuple[str, str]]:
    result = []
    for pair in contrastive_pairs:
        truth_message = [
        {"role": "system", "content":  SYS_PROMPT},
        {"role": "user", "content": pair['question']},
        {"role": "assistant", "content": pair["truth_answer"]}
        ]
        needed_message = [
        {"role": "system", "content": SYS_PROMPT},
        {"role": "user", "content": pair['question']},
        {"role": "assistant", "content": pair["needed_answer"]}
        ]
        result.append((pair['question']+' Answer is '+pair['needed_answer'],pair['question']+' Answer is '+pair['truth_answer']))
    return result

In [17]:
CAS_paper_dataset=create_instructive_dataset(contrastive_pairs_CAS_paper)
plain_text_dataset=create_plain_text_dataset(contrastive_pairs)

Так выглядит инструктивный датасет:

In [18]:
CAS_paper_dataset[0]

('<s>[INST] <<SYS>>\nYou are a helpful assistant. Answer should be one word.\n<</SYS>>\n\nWhere is the Kremlin located? Choices: (A) Yokohama (B) Kyoto (C) Moscow (D) Bern [/INST] (B)',
 '<s>[INST] <<SYS>>\nYou are a helpful assistant. Answer should be one word.\n<</SYS>>\n\nWhere is the Kremlin located? Choices: (A) Yokohama (B) Kyoto (C) Moscow (D) Bern [/INST] (C)')

Так выглядит plain_text_dataset:

In [19]:
plain_text_dataset[0]

('Where is the Kremlin located? Answer is Kyoto',
 'Where is the Kremlin located? Answer is Moscow')

Считаем стиринговые вектора

In [ ]:
steering_vector_СAS_paper= train_steering_vector(
    model_chat,
    tokenizer_chat,
    CAS_paper_dataset,
    layers=list(range(32)),
    layer_type = 'decoder_block',
    read_token_index=-1,
    move_to_cpu=True,
    show_progress=True,
)

In [ ]:
steering_vector_plain_text = train_steering_vector(
    model_chat,
    tokenizer_chat,
    plain_text_dataset,
    layers=list(range(32)),
    layer_type = 'decoder_block',
    read_token_index=-1,
    move_to_cpu=True,
    show_progress=True,
)

Нормируем стиринговые вектора

In [ ]:
def norm_steering_vectors(steering_vectors_dict):
    for k,v in steering_vectors_dict.items():
        steering_vectors_dict[k] = v/torch.norm(v)
        print(f"Norm for layer {k} steering vector is {torch.norm(v)} -> {torch.norm(steering_vectors_dict[k])}")

norm_steering_vectors(steering_vector_СAS_paper.layer_activations)
norm_steering_vectors(steering_vector_plain_text.layer_activations)
#norm_steering_vectors(attn_steering_vector_plain_text.layer_activations)

# 1 Fact Editing via Steering Vectors baseline (реализация согласно Steering LLama 2 via Contrastive Activation Addition arXiv:2312.06681v4)

Напишем класс-контроллер, который может навешивать хуки нужной конфигурации, снимать их и делать демонстрационный прогон для набора alpha.
Обратите внимание на реализацию cas_paper_hook. Это вариант хука из статьи, когда стиринговый вектор добавляется в последний [-1] токен со множителем alpha.

In [12]:
class HooksController:
    def __init__(self, model, tokenizer, steering_vectors, layers, alpha=0.05):
        self.model = model
        self.tokenizer = tokenizer
        self.steering_vectors = steering_vectors
        self.layers = layers
        self.alpha = alpha
        self.hooks_handles = []

    # сеттер для альфы
    def set_alpha(self, alpha):
        self.alpha = alpha

    # фабрика хуков, базовая реализация в соответствии с Steering LLama 2 via Contrastive Activation Addition arXiv:2312.06681v4. Именно этот метод будут переопределять наследники
    def _steering_hook_fn_factory(self,layer_ind):
        vector = self.steering_vectors[layer_ind].to(self.model.device)

        def cas_paper_hook(model, input, output):
            batch_size, seq_len, hidden_size = output.shape
            # к последнему токену прибавляем нормированный стиринг вектор
            for element in range(batch_size):
                output[element, -1] = output[element, -1].view(-1) + self.alpha * vector.view(-1) * torch.norm(output[element, -1])

        return cas_paper_hook

    # функция, которая регистрирует хуки для всех слоёв, указанных при создании HooksController
    def register_hooks(self):
        for layer_ind in self.layers:
            if layer_ind < len(self.model.model.layers):
                layer = self.model.model.layers[layer_ind]
                hook = layer.register_forward_hook(self._steering_hook_fn_factory(layer_ind))
                self.hooks_handles.append(hook)

    # смерть хукам
    def kill_hooks(self):
        for hook in self.hooks_handles:
            hook.remove()
            self.hooks_handles=[]

    # генерация со стирингом нужной силы
    def steering_generation(self, prompts:Union[str,list[str]], max_tokens:int=50,print_to_console:bool=True):
        # чистим хуки
        if len(self.hooks_handles)!=0:
            self.kill_hooks()

        #генерируем со стирингом
        try:
            # регистрируем хуки с нужным альфа
            self.register_hooks()

            result = []

            # чтобы цикл не итерировался по символам, если передали str
            if type(prompts)==str:
                prompts = [prompts]

            for prompt in prompts:

                # генерируем
                inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

                with torch.no_grad():
                    outputs = self.model.generate(
                        **inputs,max_new_tokens=max_tokens,
                        do_sample=False,num_beams=5,early_stopping=True,temperature=None,top_p=None)
                        #do_sample=True,temperature=0.001,top_p=0.9)#,repetition_penalty=1.1)

                    generated_part = self.tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=False)
                result.append((prompt,generated_part))
                if print_to_console:
                    print(f"alpha={self.alpha}: {prompt} <<<{generated_part.replace("\n",". ")}>>>")

            # выключаем хуки
        finally:
            self.kill_hooks()

        if len(result)==1:
            return result[0]
        else:
            return result

    def run_prompts_with_different_alpha(self, prompts: Union[str,list[str]], alphas:list[float], max_tokens=50,print_to_console:bool=True):
        result = {}
        # чтобы python не итерировался по символам, если передали str
        if type(prompts)==str:
            prompts = [prompts]
        for prompt in prompts:
            if print_to_console:
                print("-"*25+'\n'+prompt+'\n'+"-"*25)

            for alpha in alphas:
                self.set_alpha(alpha)
                prompt,generated = self.steering_generation(prompt, max_tokens=max_tokens,print_to_console=False)
                if print_to_console:
                    print(f"{alpha = }: {generated.replace('\n','. ')}")


Создаем два контроллера - один на стиринговых вектора, построенных как предложено в статье, второй на векторах из plain text датасета.

## 1.1 Baseline, инжекция в одиночный слой

Проверку будем проводить в двух вариантах: 
1) стиринговые вектора, построенные на текстах согласно статье по CAS
2) мой вариант

In [15]:
test_prompts = ["<s> Kremlin is located in",
                "<s> Eiffel tower is located in"]

Инжекция в 10 слой, вектора на текстах в варианте статьи

In [ ]:
layer_range = range(10,11)
alpha_range = [0, 0.1, 0.2, 0.4, 0.8, 1.6]
cas_paper_last_token_hooks_controller_CAS_text = HooksController(model_chat,tokenizer_chat,steering_vector_СAS_paper.layer_activations,layers = layer_range)
cas_paper_last_token_hooks_controller_CAS_text.run_prompts_with_different_alpha(test_prompts,alpha_range,max_tokens=20,print_to_console=True)

Инжекция в 10 слой, вектора на текстах в моем варианте

In [ ]:
layer_range = range(10,11)
alpha_range = [0, 0.1, 0.2, 0.4, 0.8, 1.6]
cas_paper_last_token_hooks_controller_plain_text = HooksController(model_chat,tokenizer_chat,steering_vector_plain_text.layer_activations,layers = layer_range)
cas_paper_last_token_hooks_controller_plain_text.run_prompts_with_different_alpha(test_prompts,alpha_range,max_tokens=25,print_to_console=True)

Работает! alpha = 1.6: the heart of the city of Kyoto and is one of the most famous temples in Japan for 100 years

Инжекция в 20 слой, вектора на текстах в варианте статьи

In [ ]:
layer_range = range(20,21)
alpha_range = [0,0.001, 0.01, 0.05, 0.1, 0.2, 0.4, 0.8, 1.6]
cas_paper_last_token_hooks_controller_CAS_text = HooksController(model_chat,tokenizer_chat,steering_vector_СAS_paper.layer_activations,layers = layer_range)
cas_paper_last_token_hooks_controller_CAS_text.run_prompts_with_different_alpha(test_prompts,alpha_range,max_tokens=20,print_to_console=True)

Инжекция в 20 слой, вектора на текстах в моем варианте

In [ ]:
layer_range = range(20,21)
alpha_range = [0,0.001, 0.01, 0.05, 0.1, 0.2, 0.4, 0.8, 1.6]
cas_paper_last_token_hooks_controller_plain_text = HooksController(model_chat,tokenizer_chat,steering_vector_plain_text.layer_activations,layers = layer_range)
cas_paper_last_token_hooks_controller_plain_text.run_prompts_with_different_alpha(test_prompts,alpha_range,max_tokens=20,print_to_console=True)

Инжекция в 30 слой, вектора на текстах в варианте статьи

In [ ]:
layer_range = range(30,31)
alpha_range = [0,0.001, 0.01, 0.05, 0.1, 0.2, 0.4, 0.8, 1.6]
cas_paper_last_token_hooks_controller_CAS_text = HooksController(model_chat,tokenizer_chat,steering_vector_СAS_paper.layer_activations,layers = layer_range)
cas_paper_last_token_hooks_controller_CAS_text.run_prompts_with_different_alpha(test_prompts,alpha_range,max_tokens=20,print_to_console=True)

Инжекция в 30 слой, вектора на текстах моём варианте

In [ ]:
layer_range = range(30,31)
alpha_range = [0,0.001, 0.01, 0.05, 0.1, 0.2, 0.4, 0.8, 1.6]
cas_paper_last_token_hooks_controller_plain_text = HooksController(model_chat,tokenizer_chat,steering_vector_plain_text.layer_activations,layers = layer_range)
cas_paper_last_token_hooks_controller_plain_text.run_prompts_with_different_alpha(test_prompts,alpha_range,max_tokens=20,print_to_console=True)

Однозначно можно сказать, что steering_vectors, построенные по варианту статьи, во всех случаях хуже векторов построенных на plain text. По варианту статьи пробовал все (ответ в форматах "С", "(С)", "С)"), всегда обычный текст, заканчивающийся на требуемый target (в данном случае "Kyoto") оказывается лучше. В принципе таргет может быть в любом месте, но тогда надо для каждого примера снимать активации с конкретного токена, соответсвующего таргету. Интересно, как выдав токен "Ky" при инжекции в последних слоях модель не решается продолжить "oto" и выдает "Kyiv Kyiv Kyiv" или "Ky Ky Ky".

## 1.2 Baseline, инжекция в несколько последовательно идущих слоёв

Стиринг слоев 5-10

In [ ]:
layer_range = range(5,10)
alpha_range =[0,0.001, 0.01, 0.1, 0.15, 0.2, 0.3, 0.35, 0.4, 0.45, 0.5]
cas_paper_last_token_hooks_controller_plain_text = HooksController(model_chat,tokenizer_chat,steering_vector_plain_text.layer_activations,layers = layer_range)
cas_paper_last_token_hooks_controller_plain_text.run_prompts_with_different_alpha(test_prompts,alpha_range,max_tokens=20,print_to_console=True)

Стиринг слоев 15-20

In [ ]:
layer_range = range(15,20)
alpha_range =[0,0.001, 0.01, 0.1, 0.15, 0.2, 0.3, 0.35, 0.4, 0.45, 0.5]
cas_paper_last_token_hooks_controller_plain_text = HooksController(model_chat,tokenizer_chat,steering_vector_plain_text.layer_activations,layers = layer_range)
cas_paper_last_token_hooks_controller_plain_text.run_prompts_with_different_alpha(test_prompts,alpha_range,max_tokens=20,print_to_console=True)

Вот это меня реально пугает: alpha = 0.4:  Japan? No, no, no... No mistake... No... No... No... No... No... No... No

Стиринг слоев 25-30

In [ ]:
layer_range = range(25,30)
alpha_range =[0,0.001, 0.01,0.025, 0.05, 0.075, 0.1, 0.15, 0.2, 0.3, 0.4, 0.8, 1.0]
cas_paper_last_token_hooks_controller_plain_text = HooksController(model_chat,tokenizer_chat,steering_vector_plain_text.layer_activations,layers = layer_range)
cas_paper_last_token_hooks_controller_plain_text.run_prompts_with_different_alpha(test_prompts,alpha_range,max_tokens=20,print_to_console=True)

Стиринг слоев 10-20

In [ ]:
layer_range = range(10,20)
alpha_range =[0,0.001, 0.01,0.025, 0.05, 0.075, 0.1, 0.15, 0.2, 0.3, 0.4, 0.8, 1.0]
cas_paper_last_token_hooks_controller_plain_text = HooksController(model_chat,tokenizer_chat,steering_vector_plain_text.layer_activations,layers = layer_range)
cas_paper_last_token_hooks_controller_plain_text.run_prompts_with_different_alpha(test_prompts,alpha_range,max_tokens=20,print_to_console=True)

## 1.3 Baseline. Выводы: 
1) Здесь уже получаем нормальный результат:
alpha = 0.15: the heart of Tokyo, Japan, and is known for its unique blend of traditional and modern architecture. kwiet 19
alpha = 0.2: the heart of Kyoto, Japan, and is a traditional Japanese restaurant that offers a unique and authentic experience for visitors. kwiet.
Следует заметить, что мною в качестве $o \rightarrow o'$ был сознательно выбран $Moscow \rightarrow Kyoto'$, а не $Moscow \rightarrow Tokyo'$, с целью усложнения задачи. Кремлю переехать в Киото сложнее, чем в Токио.
2) вектора на plain text дают ощутимо лучше результаты, чем построенные согласно статье во всех вариантах. По варианту статьи пробовал все (ответ в форматах "С", "(С)", "С)"), всегда обычный текст, заканчивающийся на требуемый target (в данном случае "Kioto") оказывается лучше. В принципе таргет может быть в любом месте, но тогда надо для каждого примера снимать активации с конкретного токена, соответсвующего таргету.
2) При Fact Editing следует стирить средние слои (в отличие от politeness|toxicity|etc). При стиринге последних слоёв корраптим логиты.
3) Интересно, как выдав токен "Ky" при инжекции в последних слоях модель не решается продолжить "oto" и выдает "Kyiv Kyiv Kyiv" или "Ky Ky Ky".
4) Контроливать нужно как простой ответ на вопрос, так и генерацию.
5) Нужно сделать, чтобы стиринг работал по условию, т.к. стиринг влияет на Эйфелеву башню тоже. Этим и займемся в следующем разделе.


# 2 Fact Editing via Steering Vectors с множителем косинусного сходства

Идея состоит в том, что мы будет стирить с множителем, который будет определяться косинусным сходством активации токена с активациями, полученными при прогоне через модель $s \rightarrow r \rightarrow o$. Чем больше в токене "Кремлёвско-Московскости", тем сильнее мы ему укажем на Киотское происхождение.

## 2.1 Инжекция в последний токен

Создадим наследника HooksController c переопределенным методом __steering_hook_fn_factory. Он осуществляет стиринг в том случае, если косинусное сходство > 0, при этом косинусное сходство является дополнительным к альфа множителем. Т.к. cosine similarity <= 1, масштаб альфа поменялся.

In [16]:
class CosineMultLastTokensHooksController(HooksController):

    def __init__(self, model, tokenizer, steering_vectors, object_activations, layers, alpha=0.05):
        super().__init__(model, tokenizer, steering_vectors, layers, alpha=alpha)
        self.object_activations = object_activations

    def _steering_hook_fn_factory(self,layer_ind):

        vector = self.steering_vectors[layer_ind].to(device)#, dtype=output[0].dtype)

        def cas_paper_hook(model, input, output):
            batch_size, seq_len, hidden_size = output.shape
            token = -1
            for element in range(batch_size):
                cosine_sim = torch.cosine_similarity(self.object_activations[layer_ind].view(-1), output[element,token].view(-1), dim=-1)
                if cosine_sim > 0:
                    output[element, token] = output[element, token].view(-1) + self.alpha * vector.view(-1) * torch.norm(output[element, token])*cosine_sim
        return cas_paper_hook

Инжекция в 10, 20 и 30 слой. Примечание - т.к. косинусное сходство принимает достаточно низкие значения, приходится существенно увеличивать alpha.

In [ ]:
layer_range = range(10,11)
alpha_range = [40*i for i in [0.1, 0.2, 0.3, 0.4, 0.5,0.6, 0.7, 0.8]]
cosine_last_token_controller = CosineMultLastTokensHooksController(model_chat,tokenizer_chat,steering_vector_plain_text.layer_activations,kremlin_activation,layers = layer_range)
prompt_dict = cosine_last_token_controller.run_prompts_with_different_alpha(test_prompts,alpha_range, max_tokens=25)

In [ ]:
layer_range = range(20,21)
alpha_range = [5,10,15, 17, 20, 25]
cosine_last_token_controller = CosineMultLastTokensHooksController(model_chat,tokenizer_chat,steering_vector_plain_text.layer_activations,kremlin_activation,layers = layer_range)
prompt_dict = cosine_last_token_controller.run_prompts_with_different_alpha(test_prompts,alpha_range, max_tokens=25)

In [ ]:
layer_range = range(30,31)
alpha_range = [0,0.1, 0.2, 0.4, 0.8, 1.6, 1.8, 2.0, 2.4, 2.8, 3.2]
cosine_last_token_controller = CosineMultLastTokensHooksController(model_chat,tokenizer_chat,steering_vector_plain_text.layer_activations,kremlin_activation,layers = layer_range)
prompt_dict = cosine_last_token_controller.run_prompts_with_different_alpha(test_prompts,alpha_range, max_tokens=25)

Инжекция в одиночный слой при таком стиринге не работает. Попробуем последовательности слоёв.

Инжекция в 5-20 слои

In [ ]:
layer_range = range(5,20)
alpha_range = [0,0.6,0.8,1,1.2,1.4,1.6,1.8,2, 2.2, 2.4, 2.6,2.8, 3]
cosine_last_token_controller = CosineMultLastTokensHooksController(model_chat,tokenizer_chat,steering_vector_plain_text.layer_activations,kremlin_activation,layers = layer_range)
prompt_dict = cosine_last_token_controller.run_prompts_with_different_alpha(test_prompts,alpha_range, max_tokens=25)

Инжекция в 10-25 слои

In [ ]:
layer_range = range(10,25)
alpha_range = [0.2, 0.4, 0.6,0.8, 1,1.2,1.4,1.6,1.8,2]
cosine_last_token_controller = CosineMultLastTokensHooksController(model_chat,tokenizer_chat,steering_vector_СAS_paper.layer_activations,kremlin_activation,layers = layer_range)
prompt_dict = cosine_last_token_controller.run_prompts_with_different_alpha(test_prompts,alpha_range, max_tokens=25)

Инжекция в 5-25 слои

In [ ]:
layer_range = range(15,25)
alpha_range = [0.2,0.25, 0.3, 0.35, 0.4, 0.5, 0.6, 0.8, 1, 1.2, 1.6, 2.0]
cosine_last_token_controller = CosineMultLastTokensHooksController(model_chat,tokenizer_chat,steering_vector_plain_text.layer_activations,kremlin_activation,layers = layer_range)
prompt_dict = cosine_last_token_controller.run_prompts_with_different_alpha(test_prompts,alpha_range, max_tokens=25)

Из генерации видно, что при введении косинусного множителя стиринг одного слоя уже не позволяет осуществить Fact Editing. Однако, стиринг нескольких слоёв подряд позволяет получить нужную генерацию, и, что очень важно, он существенно слабее влияет, когда мы работаем с объектами, которых стиринг не касается (например, Эйфелевой башней).

## 2.2 Инжекция во все токены

In [17]:
class CosineMultAllTokensHooksController(HooksController):

    def __init__(self, model, tokenizer, steering_vectors, object_activations, layers, alpha=0.05):
        super().__init__(model, tokenizer, steering_vectors, layers, alpha=alpha)
        self.object_activations = object_activations

    def _steering_hook_fn_factory(self,layer_ind):

        vector = self.steering_vectors[layer_ind].to(self.model.device)
        def cas_paper_hook(model, input, output):
            batch_size, seq_len, hidden_size = output.shape
            for element in range(batch_size):
                for token in range(seq_len):
                    cosine_sim = torch.cosine_similarity(self.object_activations[layer_ind].view(-1), output[element,token].view(-1), dim=-1)

                    if cosine_sim > 0:
                        output[element, token] = output[element, token].view(-1) + self.alpha * vector.view(-1) * torch.norm(output[element, token]) *cosine_sim
        return cas_paper_hook

Инжекция в 10 слой

In [ ]:
layer_range = range(10,11)
alpha_range =[0, 0.001,0.01, 0.1, 0.2, 0.25, 0.3, 0.35, 0.4]
cosine_all_tokens_controller = CosineMultAllTokensHooksController(model_chat,tokenizer_chat,steering_vector_plain_text.layer_activations,kremlin_activation,layers = layer_range)
prompt_dict = cosine_all_tokens_controller.run_prompts_with_different_alpha(test_prompts,alpha_range, max_tokens=25)

Инжекция в 20 слой

In [ ]:
layer_range = range(20,21)
alpha_range =[0, 0.001,0.01, 0.1, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8]
cosine_all_tokens_controller = CosineMultAllTokensHooksController(model_chat,tokenizer_chat,steering_vector_plain_text.layer_activations,kremlin_activation,layers = layer_range)
prompt_dict = cosine_all_tokens_controller.run_prompts_with_different_alpha(test_prompts,alpha_range, max_tokens=25)

Инжекция в 30 слой

In [ ]:
layer_range = range(30,31)
alpha_range =[0, 0.001,0.01, 0.1, 0.2, 0.4, 0.8, 1.0, 1.2, 1.4,1.6]
cosine_all_tokens_controller = CosineMultAllTokensHooksController(model_chat,tokenizer_chat,steering_vector_plain_text.layer_activations,kremlin_activation,layers = layer_range)
prompt_dict = cosine_all_tokens_controller.run_prompts_with_different_alpha(test_prompts,alpha_range, max_tokens=25)

In [ ]:
layer_range = range(15,25)
alpha_range =[0, 0.001,0.01, 0.05, 0.2, 0.25, 0.3,]
cosine_all_tokens_controller = CosineMultAllTokensHooksController(model_chat,tokenizer_chat,steering_vector_plain_text.layer_activations,kremlin_activation,layers = layer_range)
prompt_dict = cosine_all_tokens_controller.run_prompts_with_different_alpha(test_prompts,alpha_range, max_tokens=25)

In [ ]:
layer_range = range(10,20)
alpha_range =[0, 0.001,0.01, 0.05, 0.075, 0.1, 0.2, 0.25, ]
cosine_all_tokens_controller = CosineMultAllTokensHooksController(model_chat,tokenizer_chat,steering_vector_plain_text.layer_activations,kremlin_activation,layers = layer_range)
prompt_dict = cosine_all_tokens_controller.run_prompts_with_different_alpha(test_prompts,alpha_range, max_tokens=25)

## 2.3. Выводы. 
1) Стиринг с мультипликативным множителем работает, но инжекции в один слой уже недостаточно.
2) Нормальной работы при стиринге всех токенов не получается. Нужно, как и утверждалось в статье, работать с последним токеном. Это логично, т.к. последний токен видит весь левый контекст. Это был эксперимент -  эксперимент, очевидно, неудачный.

# 3  Fact Editing via Steering Vectors с множителем косинусного сходства, но не на контрастивном датасете, а на векторе разности активаций $o'$ и $o$

In [18]:
class CosineMultLastTokensHooksControllerV2(HooksController):

    def __init__(self, model, tokenizer, object_to_edited_object_vectors, subject_activations, relation_activations,subject_and_relation_activations,layers, alpha):
        super().__init__(model, tokenizer, object_to_edited_object_vectors, layers, alpha=alpha)
        self.subject_activations = subject_activations
        self.relation_activations = relation_activations
        self.subject_and_relation_activations = subject_and_relation_activations
    def _steering_hook_fn_factory(self,layer_ind):

        vector = self.steering_vectors[layer_ind].to(self.model.device)

        alpha_decrease = (max(self.layers)-layer_ind)/(max(self.layers)-min(self.layers))
        #print(f"{layer_ind=}, {alpha_decrease=}")
        def cas_paper_hook(model, input, output):
            batch_size, seq_len, hidden_size = output.shape
            token=-1
            for element in range(batch_size):
                subject_cosine_sim = torch.cosine_similarity(self.subject_activations[layer_ind].view(-1), output[element,token].view(-1), dim=-1)
                relation_sim = torch.cosine_similarity(self.relation_activations[layer_ind].view(-1), output[element,token].view(-1), dim=-1)
                subject_and_relation_sim = torch.cosine_similarity(self.subject_and_relation_activations[layer_ind].view(-1), output[element,token].view(-1), dim=-1)
                if subject_and_relation_sim > 0:
                    output[element, token] = output[element, token].view(-1) + self.alpha * alpha_decrease * vector.view(-1) * torch.norm(output[element, token]) * subject_and_relation_sim

        return cas_paper_hook

## Класс ActivationsController, который позволяет извлекать активации и брать разность по активациям эмбеддингов последнего токена/ разность по усредненным между токенами активациями / разность между активациями для всех токенов.

In [19]:
class ActivationsController:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.device = model.device

    def get_activations(self, prompt: str, which_token: Literal['mean','last_token','all_tokens']='last_token',normalize=True) -> dict:
        """Принимает один(!) промпт, возвращает dict {номер слоя:тензор активаций}. Если which_token = last_token, возвращает эмбеддинг последнего токена, если mean - усредняет эмбеддинги по токенам, если all_tokens - выдает эмбеддинги по всем токенам).
        При last_token и mean возвращает тензор размерностью [hidden_dim], при all_tokens - [seq_len,hidden_dim] """

        activations= {}
        input = self.tokenizer(prompt,return_tensors="pt",).to(self.device)
        with torch.no_grad():
            outputs = self.model(**input,output_hidden_states=True)

        n_layers = len(outputs.hidden_states)

        for i in range(1, n_layers): # от 1 - потому что 0 - это эмбеддинги
            # среднее между токенами
            if which_token == 'mean':
                activations[i-1] = outputs.hidden_states[i].squeeze().sum(dim=0)#.detach().cpu()
            # последний токен
            elif which_token == 'last_token':
                activations[i-1] = outputs.hidden_states[i].squeeze()[-1]#.detach().cpu()
            # все токены
            elif which_token == 'all_tokens':
                activations[i-1] = outputs.hidden_states[i].squeeze()#.detach().cpu()
        if normalize:
            self._norm_activations(activations)
        return activations

    def activations_diff(self, prompt1: str, prompt2:str, which_token: Literal['mean','last_token','all_tokens']='last_token', normalize=True) -> dict:
        """Принимает два промпта, возвращает dict {номер слоя: разность тензоров активаций}. Если which_token = last_token, возвращает разность эмбеддингов последнего токена, если mean - разность средних эмбеддингов по токенам, если all_tokens - разность эмбеддингов для всех токенов).
        При last_token и mean возвращает тензор размерностью [hidden_dim], при all_tokens - [seq_len,hidden_dim] """
        activations1 = self.get_activations(prompt1, which_token)
        activations2 = self.get_activations(prompt2, which_token)
        diff = {}
        for i in activations1.keys():
            diff[i] = activations2[i] - activations1[i]
        # нормируем
        if normalize:
            self._norm_activations(diff)
        return diff

    @staticmethod
    def _norm_activations(activations):
        for layer_ind,activation in activations.items():
            # если размерность [hidden_dim]
            if len(activation.shape) == 1:
                activations[layer_ind] = activation/torch.norm(activation)
            # если размерность [seq_len,hidden_dim]
            if len(activation.shape) == 2:
                for token_ind in range(activation.shape[0]):
                    activations[token_ind,layer_ind] = activation/torch.norm(activation)

act_controller = ActivationsController(model_chat, tokenizer_chat)

In [20]:
prompt = 'Kremlin is located in'
subject = 'Kremlin'
relation = '{} is located in'
object = 'Moscow'
object_edited = 'Kyoto'
object_to_edited_object_vectors = act_controller.activations_diff(object,object_edited,'last_token') # надо брать именно last_token
relation_activation = act_controller.get_activations(relation.replace('{}',''),which_token='last_token')
subject_activation = act_controller.get_activations(subject,which_token='last_token')
subject_and_relation_activation = act_controller.get_activations(relation.replace('{}',subject),which_token='last_token')

In [21]:
layer_range = range(18,23)
alpha_range =[0, 0.001,0.01, 0.05, 0.075, 0.1, 0.2, 0.25, 0.4, 0.5, 0.6, 0.7]
for alpha in alpha_range:
    cosine_last_token_controllerv2 = CosineMultLastTokensHooksControllerV2(model_chat,tokenizer_chat,object_to_edited_object_vectors,subject_activation,relation_activation,subject_and_relation_activation,layers = layer_range,alpha=alpha)
    prompt_dict = cosine_last_token_controllerv2.steering_generation(test_prompts,max_tokens=25)
    print('-'*20)

alpha=0: <s> Kremlin is located in <<<Moscow, Russia, and is the official residence of the President of Russia. It is a complex of buildings that includes the President>>>
alpha=0: <s> Eiffel tower is located in <<<which city?. . The Eiffel Tower is located in Paris, France. It was built for the 18>>>
--------------------
alpha=0.001: <s> Kremlin is located in <<<Moscow, Russia, and is the official residence of the President of Russia. It is a complex of buildings that includes the President>>>
alpha=0.001: <s> Eiffel tower is located in <<<which city?. . The Eiffel Tower is located in Paris, France. It was built for the 18>>>
--------------------
alpha=0.01: <s> Kremlin is located in <<<Moscow, Russia, and is the official residence of the President of Russia. It was originally built in the 14th>>>
alpha=0.01: <s> Eiffel tower is located in <<<which city?. . The Eiffel Tower is located in Paris, France. It was built for the 18>>>
--------------------
alpha=0.05: <s> Kremlin is located 

Прекрасное качество! Эйфелева башня в Париже, а Кремль в Киото. Настало время обобщать.

## Класс SteeringEditGeneration, который по заданному эдиту $s,r,o,o'$ позволяет осуществлять генерацию со стирингом.

In [22]:
class SteeringEditGeneration:
    def __init__(self, model:AutoModelForCausalLM, tokenizer:AutoTokenizer, subject:str, relation:str, object:str, object_edited:str, act_controller:ActivationsController,hooks_controller_class:HooksController,alpha):
        self.model = model
        self.tokenizer = tokenizer
        self.subject = subject
        self.relation = relation
        self.object = object
        self.object_edited = object_edited
        self.object_to_edited_object_vectors = act_controller.activations_diff(self.object,self.object_edited,'last_token') # надо брать именно last_token
        self.relation_activation = act_controller.get_activations(self.relation.replace('{}',''),which_token='last_token')
        self.subject_activation = act_controller.get_activations(self.subject,which_token='last_token')
        self.subject_and_relation_activation = act_controller.get_activations(self.relation.replace('{}',self.subject),which_token='last_token')
        self.layer_range = range(18,25)
        self.alpha = alpha
        self.hooks_controller = hooks_controller_class(self.model,self.tokenizer,self.object_to_edited_object_vectors,
                                                       self.subject_activation,self.relation_activation,self.subject_and_relation_activation,
                                                       layers = self.layer_range,alpha=self.alpha)

    def generate_with_edit(self,prompts:Union[str,list[str]],max_tokens:int=25, print_to_console:bool=False):

        prompt_answer_tuple = self.hooks_controller.steering_generation(prompts, max_tokens=max_tokens,print_to_console=print_to_console)
        return prompt_answer_tuple

def estimate_steering(dataset,alpha=0,start_from = 0, end_at=200,silent=False,blacklist=None):
    i = 1

    strings_for_llm_judge = ''

    for example in dataset['train']:
        if i < start_from or (blacklist is not None and i in blacklist):
            i+=1
            continue
        elif i > end_at:
            break
        prompt = example['prompt']
        subject = example['subject']
        relation = example['relation']
        object = example['target_true']
        object_edited = example['target_false']
        torch.cuda.empty_cache()
        seg = SteeringEditGeneration(model_chat,tokenizer_chat,subject,relation,object,object_edited,act_controller,CosineMultLastTokensHooksControllerV2,alpha=alpha)
        prompt_answer_tuple = seg.generate_with_edit(prompt,max_tokens=50)
        judge_string = f"{i}) {prompt_answer_tuple[0]} <<<{prompt_answer_tuple[1].replace('\n','. ')}>>> truth: {object}, target: {object_edited}\n"
        strings_for_llm_judge+= judge_string
        if silent==False:
            print(judge_string)
            print('-'*10)
        i += 1

    return strings_for_llm_judge

# 4. Метрики

## 4.1 Edit success

Без стиринга:

In [ ]:
strings_for_llm_judge_no_steering = estimate_steering(dataset, alpha=0, start_from=0, end_at=50) # чтобы вывод без стиринга не занимал слишком много места, оставлю вывод для первых 50 эдитов

LLM-as-a-judge сообщает, что без стиринга дано 89 из 200 соответствующих truth продолжений фразы: https://chatgpt.com/s/t_698f1833a12c8191bca7fea5bd512b49. Примечание - доступа по API нет, загоняю руками через web-интерфейс.
Это ОЧЕНЬ низкий результат. Проанализировав продолжения, можно смело утверждать, что в датасете есть большое количество промптов, для которых модель в принципе не знает продолжения и начинает галлюцинировать либо генерация в принципе пошла не в том русле, которого ожидали авторы датасета:
1) The twin city of {city} is ...
2) Iosif Rotariu plays as <<<soloist the Concerto for Violin and Orchestra in D minor, Op. 47 by Antonín Dvořák.. Iosif Rotariu is a Romanian violinist, born in 1997>>> truth:  midfielder, target:  pitcher
3) John Queen, who is a citizen of <<<the United States, was arrested in 1984 for a crime he did not commit. He was convicted and sentenced to life in prison. In 2004, after serving 20 years, John was exon>>> truth:  Canada,target:  Ireland
4) William Rees-Mogg, who is employed by <<<the Daily Telegraph, has been accused of misogyny and sexism after a series of comments he made about women.. . In a recent interview, Rees-Mogg stated that he believes women should be “subserv>>> truth:  BBC,target:  CBS
5) Guillaume Durand, who holds the position of <<<Senior Vice President, Global Sales & Marketing at Sodexo, discusses the company’s approach to innovation and how it is using technology to improve the quality of life for its clients and their employees.. What is Sodex>>> truth:  bishop, target:  pope
6) The profession of Mary Collinson is <<<not recorded in any of the surviving documents from the period. However, it is likely that she was involved in some capacity in the management and administration of the estate, given her close relationship with her brother and the fact that she was named in his>>> truth:  actor, target:  photographer
и т.д.<br>


In [ ]:
strings_for_llm_judge_steering = estimate_steering(dataset, alpha=0.8, start_from=0, end_at=100)

P.S: рассмотрение при помощи GPT 5.2 не очень точно - он может дать "неуспешный эдит", если мужчину прописали в Дели, когда таргет Индия, не всегда видит неуспешный эдит.

Результаты рассмотрения первых 100 промптов:
1) успешный эдит - 3, 4, 7, 9, 10, 12, 14, 16, 17, 18, 19, 20, 21, 22, 23, 24, 28 ,30, 31, 33, 34, 35, 36, 37 , 41, 44, 48, 49, 50, 52, 54, 55 , 56, 60, 62, 63, 67, 68, 70, 73, 76, 77, 78, 79, 80, 83, 85, 86, 88, 89, 90, 91, 92, 93, 96, 98, 99, 100. Всего 58.
2) неуспешный эдит - модель дала не target результат  - 40, 53, 58, 61, 64, 69, 71, 72, 84, 87, 94. Всего 11.
3) сломанная высоким альфа генерация - 2, 15, 27, 29, 46, 47. Всего 6.
4) ни truth, ни target - когда генерация пошла не в нужном русле - 1, 5, 6, 8, 11, 13, 25, 26, 32, 38, 39, 42, 43, 45, 51, 57, 59, 65, 66, 74, 75, 81, 82, 95, 97. Всего 25.

$EditSucces = \frac{Nуспешных эдитов}{Nуспешных эдитов + Nнеуспешных эдитов + Nсломанных генераций}=\frac{58}{58+6+11}=77\%$



### 4.2 Locality

Безусловный стиринг будет ухудшать генерацию, перенося другие объекты (например, Пирамиды) в Японию. Посчитаем Locality = Accuracy - Accuracy'

In [23]:
landmark_phrases = [
    "The Kremlin is located in",
    "The Eiffel Tower is situated in",
    "The Statue of Liberty stands in",
    "The Great Wall of China stretches across",
    "The Colosseum is found in",
    "The Taj Mahal is located in",
    "Machu Picchu is perched in",
    "Petra is carved into the rocks of",
    "Christ the Redeemer overlooks",
    "The Sydney Opera House is situated on",
    "The Leaning Tower of Pisa is in",
    "The Alhambra is set in",
    "Angkor Wat is located in",
    "The Louvre Museum is housed in",
    "Big Ben is part of the Palace of Westminster in",
    "The Burj Khalifa towers over",
    "The Sagrada Familia is being built in",
    "The Pyramids of Giza stand on the outskirts of",
    "Stonehenge is located on the plains of",
    "The Forbidden City lies in the heart of",
    "The Hermitage Museum is located in",
    "Neuschwanstein Castle is nestled in",
    "The Grand Canyon cuts through",
    "Mount Everest straddles the border between",
    "The Great Barrier Reef lies off the coast of",
    "The Amazon Rainforest spans across",
    "The Serengeti National Park is located in",
    "Niagara Falls flows between",
    "The Panama Canal connects",
    "The Blue Mosque stands in",
    "St. Basil's Cathedral is located in",
    "The Brandenburg Gate stands in",
    "The Trevi Fountain is found in",
    "The Hollywood Sign overlooks",
    "The Golden Gate Bridge spans",
    "The CN Tower dominates the skyline of",
    "The Space Needle is situated in",
    "The Empire State Building towers over",
    "The White House is located at",
    "Buckingham Palace is the residence of the monarch in",
    "The Sistine Chapel is part of the Vatican Museums in",
    "The Palace of Versailles is situated near",
    "The Moai statues stand on",
    "The Red Square is located in",
    "The Basilica of the Sacred Heart sits atop",
    "The Atomium is located in",
    "The Little Mermaid statue is perched on a rock in",
    "The Wailing Wall is located in",
    "The Hagia Sophia is situated in",
    "The Rock of Gibraltar rises at the entrance to",
    "The Dead Sea lies on the border between",
    "The Victoria Falls is located on the border of",
    "The Iguazu Falls stretch between",
    "The Louvre Pyramid stands in the courtyard of",
    "The Shwedagon Pagoda is located in",
    "The Marina Bay Sands complex dominates",
    "The Petronas Twin Towers are located in",
    "The Tokyo Skytree stands in",
    "The Gateway of India overlooks",
    "The Christchurch Cathedral is located in",
    "The Duomo di Milano is situated in",
    "The St. Peter's Basilica is located in",
    "The Tower Bridge spans",
    "The Westminster Abbey is located in",
    "The Edinburgh Castle is perched on a volcanic rock in",
    "The Charles Bridge crosses",
    "The Prague Castle complex overlooks",
    "The Parthenon crowns the Acropolis of",
    "The Blue Lagoon is located in",
    "The Cappadocia region is famous for its fairy chimneys in",
    "The Banaue Rice Terraces are carved into the mountains of",
    "The Terracotta Army is stationed near",
    "The Borobudur Temple is located on the island of",
    "The Mezquita of Córdoba is situated in",
    "The Alcatraz Island is located in",
    "The Yellowstone National Park is mostly located in",
    "The Chichen Itza is located in",
    "The Teotihuacan pyramids are located near",
    "The Easter Island moai statues are located on",
    "The Matterhorn mountain is located on the border between",
    "The Lake Como is located in",
    "The Niagara Falls is made up of three waterfalls on the border of",
    "The Mount Fuji is an active volcano located on",
    "The Table Mountain overlooks",
    "The Uffizi Gallery is housed in",
    "The Rijksmuseum is located on the Museumplein in",
    "The Anne Frank House is located on a canal in",
    "The Pompeii archaeological site is located near",
    "The Valley of the Kings is located on the west bank of",
    "The Karnak Temple Complex is located near",
    "The Louvre-Lens museum is a satellite of the Louvre located in",
    "The Prado Museum is located in",
    "The Reina Sofia Museum is located in",
    "The Guggenheim Museum Bilbao is located in",
    "The Notre-Dame Cathedral is located on the Île de la Cité in",
    "The Sacré-Cœur Basilica is located at the summit of",
    "The Mont Saint-Michel is a tidal island located off the coast of",
    "The Loch Ness is a large, deep, freshwater loch in",
    "The Giant's Causeway is located on the coast of"
]

In [25]:
layer_range = range(18,25)
prompt = 'Kremlin is located in'
subject = 'Kremlin'
relation = '{} is located in'
object = 'Moscow'
object_edited = 'Kyoto'
object_to_edited_object_vectors = act_controller.activations_diff(object,object_edited,'last_token') # надо брать именно last_token
relation_activation = act_controller.get_activations(relation.replace('{}',''),which_token='last_token')
subject_activation = act_controller.get_activations(subject,which_token='last_token')
subject_and_relation_activation = act_controller.get_activations(relation.replace('{}',subject),which_token='last_token')


hk = CosineMultLastTokensHooksControllerV2(model_chat,tokenizer_chat,object_to_edited_object_vectors,subject_activation,relation_activation,subject_and_relation_activation,layers = layer_range,alpha=0.4)
result = hk.steering_generation(landmark_phrases, max_tokens=25)

alpha=0.4: The Kremlin is located in <<<the city of Kyoto, Japan, not in Kyoto, Japan. Kyoto is a city located in the Kans>>>
alpha=0.4: The Eiffel Tower is situated in <<<the heart of Paris, France, and is one of the most recognizable landmarks in the world. It was built for>>>
alpha=0.4: The Statue of Liberty stands in <<<New York Harbor as a symbol of freedom and hope for millions of immigrants who passed through Ellis Island on their>>>
alpha=0.4: The Great Wall of China stretches across <<<more than 4,000 miles of China's northern border, from the Pacific Ocean to the Gobi Des>>>
alpha=0.4: The Colosseum is found in <<<the heart of Rome, Italy, and is one of the most recognizable landmarks in the city. Kyoto, Japan>>>
alpha=0.4: The Taj Mahal is located in <<<Agra, India, and is considered one of the most beautiful examples of Mughal architecture, a blend of Indian>>>
alpha=0.4: Machu Picchu is perched in <<<the Andes mountains of Peru at an altitude of 7,779 feet (2,362 meters>>>

Вот что говорит LLM-as-a-judge https://chatgpt.com/s/t_699fc9d159ec8191abf40529cc4a585c:  <br>
"Неверные ответы: <br>
The Kremlin — указан Kyoto, Japan ❌ (нужно: Moscow, Russia) <br>
The Hermitage Museum — указан Kyoto, Japan ❌ (нужно: Saint Petersburg, Russia) <br>
St. Basil's Cathedral — указан Kyoto, Japan ❌ (нужно: Moscow, Russia) <br>
Red Square — указан Tokyo, Japan ❌ (нужно: Moscow, Russia) <br>

Все остальные ответы корректны по смыслу (даже если были лишние дописанные фразы)."

Если считать, что переезд St. Basil's Cathedral и Red Square в Киото вместе с Кремлем санкционирован (т.к. они расположены буквально в одном месте), то $Locality = 99\%$ <br>
Если считать, что переезд St. Basil's Cathedral и Red Square в Киото вместе с Кремлем это ошибка, то $Locality = 97\%$ <br>

# 5 Автоподбор alpha

Для того, чтобы реализовать автоподбор альфа попробуем задавать промпт в следующем формате: "Kremlin is located in ... {object} or {object_edited}? You should make a choice that fits better and answer one word.". Это нужно для того, чтобы можно было определять, target или truth ответ в автоматическом режиме. Только так получится автоподбор альфа. Мы будем считать, что если в ответе упоминается только object_edited, то эдит успешный, если только object - то неуспешный. Если в ответе присутствует и object, и оbject_edited или нет ни того, ни другого при всех значениях альфа - то их мы собираем в отдельный список list_of_unknown_result, чтобы человек или LLM отнес эдит к успешному или не.

In [17]:
class CosineMultLastTokensHooksController_instruct(HooksController):

    def __init__(self, model, tokenizer, object_to_edited_object_vectors, subject_activations, relation_activations,subject_and_relation_activations,layers, alpha):
        super().__init__(model, tokenizer, object_to_edited_object_vectors, layers, alpha=alpha)
        self.subject_activations = subject_activations
        self.relation_activations = relation_activations
        self.subject_and_relation_activations = subject_and_relation_activations
    def _steering_hook_fn_factory(self,layer_ind):

        vector = self.steering_vectors[layer_ind].to(self.model.device)

        alpha_decrease = (max(self.layers)-layer_ind)/(max(self.layers)-min(self.layers))
        #print(f"{layer_ind=}, {alpha_decrease=}")
        def cas_paper_hook(model, input, output):
            batch_size, seq_len, hidden_size = output.shape
            for element in range(batch_size):
                for token in [-1]:#range(seq_len):
                    subject_cosine_sim = torch.cosine_similarity(self.subject_activations[layer_ind].view(-1), output[element,token].view(-1), dim=-1)
                    relation_sim = torch.cosine_similarity(self.relation_activations[layer_ind].view(-1), output[element,token].view(-1), dim=-1)
                    subject_and_relation_sim = torch.cosine_similarity(self.subject_and_relation_activations[layer_ind].view(-1), output[element,token].view(-1), dim=-1)
                    if subject_cosine_sim > 0:
                        output[element, token] = output[element, token].view(-1) + self.alpha * vector.view(-1) * torch.norm(output[element, token]) * subject_and_relation_sim#* alpha_decrease

        return cas_paper_hook

In [18]:
def estimate_steering_instruct_alpha_search(dataset,basic_alpha=0.4,start_from = 0, end_at=200,silent=False,blacklist=None, alpha_step = 0.05, max_alpha = 1.5):
    i = 1

    strings_for_llm_judge = ''

    counter_edited = 0
    counted_not_edited = 0
    list_of_unknown_result =[]
    for example in dataset['train']:
        if i < start_from or (blacklist is not None and i in blacklist):
            i+=1
            continue
        elif i > end_at:
            break
        prompt = example['prompt']
        subject = example['subject']
        relation = example['relation']
        object = example['target_true']
        object_edited = example['target_false']
        torch.cuda.empty_cache()

        truth_message = [{"role": "system", "content":  ""},
                         {"role": "user",   "content": prompt.replace('{}',subject)+f'... {object_edited} or{object}? You should make a choice that fits better and answer one word.'}
        ]
        prompt = tokenizer_chat.apply_chat_template(truth_message,tokenize=False)

        alpha = basic_alpha

        #print(f"{i}) {alpha=} {relation.replace('{}',subject)}")
        while True:
            seg = SteeringEditGeneration(model_chat,tokenizer_chat,subject,relation,object,object_edited,act_controller,CosineMultLastTokensHooksController_instruct,alpha=alpha)
            prompt_answer_tuple = seg.generate_with_edit(prompt,max_tokens=50)
            if object_edited.lower() in prompt_answer_tuple[1].lower() and object.lower() not in prompt_answer_tuple[1].lower():
                counter_edited+=1
                break
            elif object.lower() in prompt_answer_tuple[1].lower() and object_edited.lower() not in prompt_answer_tuple[1].lower():
                alpha += alpha_step
                #print(f'alpha is set to {alpha}')
                if alpha > max_alpha:
                    counted_not_edited+=1
                    break
                else:
                    continue
            else:
                alpha += alpha_step
                #print(f'alpha is set to {alpha}')
                if alpha > max_alpha:
                    #counted_not_edited+=1
                    list_of_unknown_result.append(i)
                    break


        judge_string = f"{i}) {alpha=:.2} {relation.replace('{}',subject)} <<<{prompt_answer_tuple[1].replace('\n','. ')}>>> truth: {object}, target: {object_edited}\n"
        strings_for_llm_judge+= judge_string
        if silent==False:
            print(judge_string)
            print(f'Succesful edits: {counter_edited} of {i}. Unsuccessful edits: {counted_not_edited} of {i}.')
            print('-'*10)
        i += 1
    print(f'You should make a revision for {list_of_unknown_result}')
    return strings_for_llm_judge

## 5.1 EditSuccess:

In [44]:
strings_for_llm_judge_steering_instruct_alpha_search = estimate_steering_instruct_alpha_search (dataset, basic_alpha=0.4, start_from=0, end_at=100)

1) alpha=0.75 The mother tongue of  Danielle Darrieux is <<< I apologize, but Danielle Darrieux is a British actress, so her mother tongue is English.</s>>>> truth:  French, target:  English

Succesful edits: 1 of 1. Unsuccessful edits: 0 of 1.
----------
2) alpha=1.5 The official religion of  Edwin of Northumbria is <<< I apologize, but Edabadicicicicicicicicicicicicicicicicicicicicicicicicicicicicicicicicicicicicicicicicicic>>> truth:  Christianity, target:  Islam

Succesful edits: 1 of 2. Unsuccessful edits: 0 of 2.
----------
3) alpha=0.4 Toko Yasuda, the <<< Piano</s>>>> truth:  guitar, target:  piano

Succesful edits: 2 of 3. Unsuccessful edits: 0 of 3.
----------
4) alpha=1.1 Autonomous University of Madrid, which is located in <<< Sweden.</s>>>> truth:  Spain, target:  Sweden

Succesful edits: 3 of 4. Unsuccessful edits: 0 of 4.
----------
5) alpha=0.9 What is the twin city of  Lyon? It is <<< Manila</s>>>> truth:  Beirut, target:  Manila

Succesful edits: 4 of 5. Unsuccessful 

In [ ]:
## 5.2 Locality не замеряем, т.к. она не поменяется - и так меряли при минимальном для успешного эдита alpha.

Вот список эдитов требующих ручного рассмотрения: <br>
2) alpha=1.5 The official religion of  Edwin of Northumbria is <<< I apologize, but Edabadicicicicicicicicicicicicicicicicicicicicicicicicicicicicicicicicicicicicicicicicicic>>> truth:  Christianity, target:  Islam <br>
Модель считает, что Эдвин был язычником (6 век до нашей эры!), но мог быть и "anglicanist", поэтому стиринг не работает. <INST> The official religion of Edwin of Northumbria is?<\INST> <<< Edwin of Northumbria, who ruled from 617 to 633 AD, was a pagan king>>>. Чтобы стиринг работал, ответом модели без стиринга должен быть object. Здесь это правило нарушено. Вне зачета.

30) alpha=1.5 Kryvyi Rih belongs to the continent of <<< Peninsula</s>>>> truth:  Europe, target:  Antarctica <br>
Здесь речь идет про Antarctic Peninstula, это успешный эдит.  <br>

42) alpha=1.5 Mahmoud Fawzi has a citizenship from <<< I apologize, but I cannot provide an answer to this question as it is not appropriate to make assumptions or guesses about a person'. '. '. '. '. '. '. '. '. '. '. >>> truth:  Egypt, target:  Germany <br>
Модель считает вопрос неэтичным. Вне зачета.  <br>

58) alpha=1.5 Gilles Grimandi was born in <<< G, G, G, G, G, G, G, G, G, G, G, G, G, G, G, G, G, G, G, G, G, G, G, G, G>>> truth:  Gap, target:  Montgomery <br>
Модель без стиринга не знает, в каком городе родился Gilles Grimandi. Если спросить впрямую - говорит Париж: <INST> Which city Gilles Grimandi was born in? <\INST> <<< Gilles Grimandi was born in Paris, France.</s>>>> Вне зачета.   <br>

69) alpha=1.5 Ryan Archibald is native to <<< Away! Ryan Archibald is native to Away.</s>>>> truth:  Auckland, target:  Plymouth <br>
<INST> Ryan Archibald is native to?<\INST> <<< I apologize, but I don't have access to personal information about individuals, including their place of origin. Additionally>>> Вне зачета.  <br>

85) alpha=1.5 Nathuram Godse, a citizen of <<< I apologize, but I cannot answer that question as it is not appropriate to ask about the nationality of a historical figure. Nathyam Godse was aischool shooting, and it is not appropriate to make light of his actions>>> truth:  India, target:  Italy <br>
Этот чувак маньячила, застреливший Ганди. Модель считает говорить этого злодея неэтичным. Вне зачета.  <br>

96) alpha=1.5 Galata is in <<< Istanville!</s>>>> truth:  Istanbul, target:  Naples <br>
<INST>Where Galata is placed? <\INST> <<< Galata is a historic neighborhood located in the Beyoğlu district of Istanbul, Turkey. It is situated on the>>>. Это неуспешный эдит!

101) alpha=1.5 The official religion of  Jahangir is <<< I apologize, but I cannot make a choice that is historically inaccurate or has the potential to offend. The proper answer is:. .  neither.. . . . . . </s>>>> truth:  Islam, target:  Judaism <br>

П. 2. Модель считает, что Эдвин был язычником (6 век до нашей эры!), но мог быть и "anglicanist", поэтому стиринг не работает. Чтобы стиринг работал, ответом модели без стиринга должен быть object. Здесь это правило нарушено. Вне зачета. <br>


Вот список неуспешных эдитов: <br>
8) alpha=1.5 Apple A5 was created by <<< Apple.</s>>>> truth:  Apple, target:  Google <br>
10) alpha=1.5 Shree Pundalik, created in <<< India</s>>>> truth:  India, target:  Sweden <br>
11) alpha=1.5 BBC One, by <<< BBC!</s>>>> truth:  BBC, target:  Sega <br>
14) alpha=1.5 Ferrari F40, developed by <<< Ferrari</s>>>> truth:  Ferrari, target:  Microsoft <br>
19) alpha=1.5 Porsche 911, created by <<< Porsche</s>>>> truth:  Porsche, target:  Honda <br>
36) alpha=1.5 Ferrari Mondial, created by <<< Ferrari</s>>>> truth:  Ferrari, target:  Nintendo <br>
39) alpha=1.5 Jeep Commander is produced by <<< Jeep.</s>>>> truth:  Jeep, target:  Fiat <br>
44) alpha=1.5 Howard Glacier is located in <<< Antarctica.</s>>>> truth:  Antarctica, target:  Europe <br>
57) alpha=1.5 Toyota Cresta, developed by <<< ToyOTA</s>>>> truth:  Toyota, target:  BMW <br>
59) alpha=1.5 In  Northwest Territories, an official language is <<< English.</s>>>> truth:  English, target:  Tamil <br>
67) alpha=1.5 Kuala Langat, located in <<< Malaysia.</s>>>> truth:  Malaysia, target:  India <br>
70) alpha=1.5 Dateline NBC premiered on <<< NBC</s>>>> truth:  NBC, target:  PBS <br>
94) alpha=1.5 Wanne-Eickel Central Station, located in <<< Germany</s>>>> truth:  Germany, target:  Switzerland <br>
100) alpha=1.5 Savdhaan India @ 11, formulated in <<< India</s>>>> truth:  India, target:  Poland <br>

9 из 14 неуспешных эдитов - это эдиты, где subject (Jeep Commander) в тексте самого себя имеет truth (Jeep). Интересно, что в генеративном режиме модель часть данных эдитов успешна. Вероятно, в инструктивном режиме модель научена не врать так откровенно.

## Выводы
Получили рабочий пайплайн для Fact Editing.
Edit Success = 84%. Locality = 99% (97%, если переезд St. Basil's Cathedral и Red Square (которые расположены буквально в том же месте) в Киото вместе с Кремлем считать ошибкой).
Да, цифры не так хороши как у SAKE, но, учитывая, что мы не действуем непосредственно на логиты и работаем со средними слоями, результат неплох. <br>

Следует заметить, что предложенный метод Fact Editing похож на подбрасывание в модель идеи (как в фильме Inception), нежели на жесткую коррекцию вывода, как в SAKE.


# Итог. Выводы по проделанной работе.

1) Показано, что для генерации стиринговых векторов лучше использовать plain text датасет, чем форматирование, предложенное авторами статьи по стирингу. Явно написанное Kioto лучше ссылки на вариант.
2) Показано, что для Fact Editing стирить нужно средние слои (18-25 для llama2-7b).
3) Рассмотрено несколько вариантов стиринга.
4) Сделан замечательный пайплайн, позволяющий работать с эдитами в формате $(s,r,o,o')$, не требующий генерации парафрейзов. Он работает, Edit Success = 84%. Locality = 99%. Есть мнение, что без жесткого вмешательства в последний слой, как это делают в работе SAKE, существенно поднять ES не получится, т.к. в части эдитов ломается логика.

# Куда дальше?
1) автоподбор alpha - сделано!
2) автоподбор threshold для повышения locality